In [ ]:
import requests
from dotenv import load_dotenv
import os
import pandas as pd
import rasterio
import numpy as np
import glob

load_dotenv('../envs/dev.env')

token = os.getenv("HCDP_ADMIN_TOKEN")
rainfall_tif_dir = ""

headers = {
    "Authorization": f"Bearer {token}"
}

In [10]:
# date, startDate, endDate, island, division_type, and name. For  island, division_type, and name you can provide multiple instances of each or a comma separated list to filter by more than one (e.g. ?division_type=moku&division_type=island or  ?division_type=moku,island)
#Viewing data

url = "https://api.hcdp.ikewai.org/mesonet/climate_report/rainfall_stats"
# rainfall_stats, temperature_stats, drought_stats, rainfall_historical, or temperature_historical
query_params = {
    "division_type": "Statewide"
    # "startDate": "2026-01-01",
    # "endDate": "2026-03-01"
}

response = requests.get(url, params=query_params, headers=headers)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    df_filtered = pd.DataFrame(response.json())
    display(df_filtered)
else:
    print(f"Response Text: {response.text}")

Status Code: 200


,island,division_type,name,date,mean,anomaly,pchange,rank,ytd_pnormal
0,Statewide,Statewide,Statewide,2026-02-01T00:00:00.000Z,-9999,-9999,-9999,-9999,-9999
1,Statewide,Statewide,Statewide,2026-03-01T00:00:00.000Z,22.289707995859303,15.436666994520824,225.25280370430974,2,206.9470762479052


In [ ]:
def convert_units(value, dataset):
  """Convert rainfall mm to inches and temperature C to F"""
  if value is None or np.isnan(value):
      return np.nan
  if dataset == "rainfall":
      return value / 25.4
  elif dataset == "temperature":
      return (value * 9/5) + 32
  return value

def get_statewide_stats(dataset, year, month):
  """Compute statewide mean, anomaly, percent change, and drought percentage."""
  climo_file = os.path.join(local_dep_dir, f"climo/{dataset}/{dataset}_1991-2020_{month:02d}.tif")

  print(f"\n--- Processing statewide ({dataset}) ---")

  with rasterio.open(climo_file) as src:
      clim = src.read(1).astype(float)
      clim = np.where(src.nodata == src.read(1), np.nan, clim)
      climo_mean = convert_units(np.nanmean(clim), dataset)

  all_records = []
  for tif in sorted(glob.glob(os.path.join(local_dep_dir, dataset, f"{dataset}_*_{month:02d}.tif"))):
      parts = os.path.basename(tif).replace(".tif", "").split("_")
      curr_year, curr_month = parts[1], parts[2]
      curr_date = f"{curr_year}-{curr_month}"

      with rasterio.open(tif) as src:
          arr = src.read(1).astype(float)
          arr = np.where(arr == src.nodata, np.nan, arr)
          mean_val = convert_units(np.nanmean(arr), dataset)

          if dataset == "temperature":
              # Suppress the warning if the array is entirely NaNs
              with np.errstate(all='ignore'):
                  max_val = convert_units(np.nanmax(arr), dataset)

      anomaly = mean_val - climo_mean
      pchange = ((mean_val - climo_mean) / climo_mean) * 100 if dataset == "rainfall" else anomaly

      record = {
          "date": curr_date,
          "mean": mean_val,
          "anomaly": anomaly,
          "pchange": pchange,
      }
      if dataset == "temperature":
          record["max"] = max_val

      all_records.append(record)

  df = pd.DataFrame(all_records)

  df["rank"] = df["anomaly"].rank(method="min", ascending=False)
  num_rows = len(df)
  latest = df[df["date"] == f"{year}-{month:02d}"].copy()
  if latest.empty:
      print(f"No data found for {year}-{month:02d}")
      return

  if dataset == "rainfall":
      current_ytd_path = make_ytd(year, month)
      climo_ytd_path = os.path.join(local_dep_dir, "climo", "rainfall_ytd", f"YTD_rain_month_{month:02d}.tif")

      with rasterio.open(current_ytd_path) as src:
          curr_arr = src.read(1).astype(float)
          curr_arr = np.where(curr_arr == -9999, np.nan, curr_arr)
          curr_mean = np.nanmean(curr_arr)

      with rasterio.open(climo_ytd_path) as src:
          climo_arr = src.read(1).astype(float)
          climo_arr = np.where(climo_arr == -9999, np.nan, climo_arr)
          climo_mean = np.nanmean(climo_arr)

      if not np.isnan(curr_mean) and not np.isnan(climo_mean) and climo_mean != 0:
          pnormal = (curr_mean / climo_mean) * 100
      else:
          pnormal = np.nan

      latest["ytd_pnormal"] = pnormal

  latest["island"] = "Statewide"
  latest["division_type"] = "Statewide"
  latest["name"] = "Statewide"

  base_cols = ["island", "division_type", "name", "date", "mean", "anomaly", "pchange", "rank"]

  if dataset == "rainfall":
      base_cols.append("ytd_pnormal")
  elif dataset == "temperature":
      base_cols.append("max")

  latest_df = latest[base_cols]

  out_csv = os.path.join(output_dir, dataset, f"statewide_{dataset}_stats.csv")
  latest_df.to_csv(out_csv, index=False)
  print(f"Saved {out_csv}")
  return num_rows


In [9]:
url = "https://api.hcdp.ikewai.org/mesonet/climate_report/rainfall_stats"


# const STAT_TABLE_DATA = {
#   rainfall_stats: ['division_full', 'date', 'mean', 'anomaly', 'pchange', 'rank', 'ytd_pnormal'],
#   temperature_stats: ['division_full', 'date', 'mean', 'anomaly', 'pchange', 'rank', 'max'],
#   drought_stats: ['division_full', 'date', 'd4', 'd3', 'd2', 'd1', 'd0', 'near_normal', 'w0', 'w1', 'w2', 'w3', 'w4'],
#   rainfall_historical: ['division_full', 'date', 'value'],
#   temperature_historical: ['division_full', 'date', 'value']
# };

payload = {
    "overwrite": True,
    "data": [
        ["Statewide","Statewide","Statewide", "2026-02-01", -9999, -9999, -9999, -9999, -9999]
    ]
}

response = requests.post(url, json=payload, headers=headers)

# 4. Print the result to see if it worked
print(f"Status Code: {response.status_code}")
print(f"Response Text: {response.text}")

Status Code: 200
Response Text: {"modified":1}
